### The purpose of this is to take people involved in movies and turn them into scores

The key idea is by looking at the median and average revenue of the movies people are in you can turn that into a score for them. There are two types of people those that are actors and those involved in production. Then the data is pivoted such that instead of multiple copies of the movie, there are just scores for production and actors in them, representing their starpower.

In [118]:
import pandas as pd
import numpy as np
df = pd.read_csv("movie-data/combined_df.csv")
movies = df[df["revenue"] > 0]
movies.head()


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,tagline,genres,production_companies,production_countries,spoken_languages,keywords,tconst,nconst,category,primaryName
0,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0000138,actor,Leonardo DiCaprio
1,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0330687,actor,Joseph Gordon-Levitt
2,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0680983,actor,Elliot Page
3,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0913822,actor,Ken Watanabe
4,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0362766,actor,Tom Hardy


In [119]:
actor_roles = ['actor', 'actress', "self"]
df_actors = movies[movies['category'].isin(actor_roles)].copy()
production_roles = ['director', 'producer', 'writer']
df_production = movies[movies['category'].isin(production_roles)].copy()

print("Actors/Actresses:", df_actors.shape)
print("Production:", df_production.shape)


Actors/Actresses: (164265, 25)
Production: (85897, 25)


In [120]:
# Compute profit per movie
df_actors['profit'] = df_actors['revenue'] - df_actors['budget']

# Aggregate profit data per actor
agg_actors = df_actors.groupby(['nconst', 'primaryName']).agg(
    avg_profit=('profit', 'mean'),
    med_profit=('profit', 'median'),
    num_titles=('title', 'nunique')
).reset_index()

agg_actors['composite'] = (agg_actors['avg_profit'] + agg_actors['med_profit']) / 2

# Punish low number of titles: scale weight if num_titles < 4; otherwise weight=1
agg_actors['weight'] = agg_actors['num_titles'].apply(lambda x: x / 4 if x < 4 else 1)

agg_actors['final_composite'] = agg_actors['composite'] * agg_actors['weight']

mean_comp = agg_actors['final_composite'].mean()
std_comp = agg_actors['final_composite'].std()
agg_actors['z_score'] = (agg_actors['final_composite'] - mean_comp) / std_comp

# Transform z-score to final score; allow negative values for flops
agg_actors['final_score'] = 100 * agg_actors['z_score']

agg_actors.sort_values('final_score', ascending=False)[
    ['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 
     'weight', 'final_composite', 'z_score', 'final_score']
].head()


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_composite,z_score,final_score
48650,nm1569276,Chadwick Boseman,11,8.387375e+08,905046416.0,8.718920e+08,1.0,8.718920e+08,34.223513,3422.351319
51789,nm1853544,Pierre Coffin,4,7.732569e+08,903090397.5,8.381737e+08,1.0,8.381737e+08,32.889160,3288.915968
317,nm0000355,Anthony Daniels,9,6.404379e+08,737000000.0,6.887189e+08,1.0,6.887189e+08,26.974704,2697.470438
37629,nm1019674,Sala Baker,4,5.289598e+08,778368364.0,6.536641e+08,1.0,6.536641e+08,25.587459,2558.745858
26101,nm0641063,Dean O'Gorman,4,5.465927e+08,707209894.0,6.269013e+08,1.0,6.269013e+08,24.528360,2452.835999


In [121]:
print("Aggregated Actors/Actresses:")
agg_actors.sort_values('final_score', ascending=False)[['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 'weight', 'final_score']].head(15)

Aggregated Actors/Actresses:


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_score
48650,nm1569276,Chadwick Boseman,11,8.387375e+08,905046416.0,8.718920e+08,1.00,3422.351319
51789,nm1853544,Pierre Coffin,4,7.732569e+08,903090397.5,8.381737e+08,1.00,3288.915968
317,nm0000355,Anthony Daniels,9,6.404379e+08,737000000.0,6.887189e+08,1.00,2697.470438
37629,nm1019674,Sala Baker,4,5.289598e+08,778368364.0,6.536641e+08,1.00,2558.745858
26101,nm0641063,Dean O'Gorman,4,5.465927e+08,707209894.0,6.269013e+08,1.00,2452.835999
60864,nm3269138,Dana Gaier,3,7.703313e+08,894761885.0,8.325466e+08,0.75,2442.976921
45672,nm1388927,Miranda Cosgrove,3,7.703313e+08,894761885.0,8.325466e+08,0.75,2442.976921
59906,nm3094377,Willow Shields,4,6.195479e+08,624875717.5,6.222118e+08,1.00,2434.278016
36107,nm0942247,Bonnie Wright,4,5.339891e+08,694132532.5,6.140608e+08,1.00,2402.021754
63901,nm3918035,Zendaya,6,6.864883e+08,527005800.5,6.067471e+08,1.00,2373.078631


In [122]:
print(agg_actors[agg_actors["primaryName"] == "Brad Pitt"])

       nconst primaryName    avg_profit  med_profit  num_titles     composite  \
73  nm0000093   Brad Pitt  1.155123e+08  90845033.0          44  1.031787e+08   

    weight  final_composite   z_score  final_score  
73     1.0     1.031787e+08  3.802793   380.279298  


In [123]:
# Compute profit per movie for production personnel
df_production['profit'] = df_production['revenue'] - df_production['budget']

# Aggregate profit data per production person
agg_production = df_production.groupby(['nconst', 'primaryName']).agg(
    avg_profit=('profit', 'mean'),
    med_profit=('profit', 'median'),
    num_titles=('title', 'nunique')
).reset_index()

agg_production['composite'] = (agg_production['avg_profit'] + agg_production['med_profit']) / 2

# Apply a weight that scales if num_titles < 4; otherwise, weight = 1
agg_production['weight'] = agg_production['num_titles'].apply(lambda x: x / 4 if x < 4 else 1)

agg_production['final_composite'] = agg_production['composite'] * agg_production['weight']

mean_comp = agg_production['final_composite'].mean()
std_comp = agg_production['final_composite'].std()
agg_production['z_score'] = (agg_production['final_composite'] - mean_comp) / std_comp

agg_production['final_score'] = 100 * agg_production['z_score']

agg_production.sort_values('final_score', ascending=False)[
    ['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 
     'weight', 'final_composite', 'z_score', 'final_score']
].head()


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_composite,z_score,final_score
9501,nm0484457,Jon Landau,6,1.138348e+09,1.047615e+09,1.092982e+09,1.0,1.092982e+09,27.596469,2759.646929
23241,nm1601644,Jennifer Lee,4,8.640651e+08,1.124219e+09,9.941421e+08,1.0,9.941421e+08,25.078253,2507.825274
3010,nm0118333,Chris Buck,4,7.563961e+08,1.124219e+09,9.403076e+08,1.0,9.403076e+08,23.706672,2370.667165
24505,nm1853544,Pierre Coffin,4,8.484312e+08,9.231572e+08,8.857942e+08,1.0,8.857942e+08,22.317796,2231.779565
20534,nm1273099,Erik Sommers,5,8.627477e+08,9.053391e+08,8.840434e+08,1.0,8.840434e+08,22.273189,2227.318929


### Ok now here you is where you add the columns you just calculated

In [ ]:
movies_with_actor_scores = pd.merge(movies, agg_actors[['nconst', 'final_score']],
                                    on='nconst', how='left', suffixes=('', '_actor'))
actor_stats = movies_with_actor_scores[movies_with_actor_scores['category'].isin(actor_roles)] \
    .groupby('tconst')['final_score'] \
    .agg(actor_avg='mean', actor_med='median', actor_dev='std') \
    .reset_index()


movies_with_prod_scores = pd.merge(movies, agg_production[['nconst', 'final_score']],
                                   on='nconst', how='left', suffixes=('', '_prod'))

prod_stats = movies_with_prod_scores[movies_with_prod_scores['category'].isin(production_roles)] \
    .groupby('tconst')['final_score'] \
    .agg(production_avg='mean', production_med='median', production_dev='std') \
    .reset_index()

#prod_stats.head(10)
#actor_stats.head(10)
#movies_with_actor_scores[["title", "category", "primaryName", "final_score"]].head(10)
movies_with_prod_scores[["title", "category", "primaryName", "final_score"]].head(10)


,title,category,primaryName,final_score
0,Inception,actor,Leonardo DiCaprio,576.493568
1,Inception,actor,Joseph Gordon-Levitt,121.875687
2,Inception,actor,Elliot Page,155.768450
3,Inception,actor,Ken Watanabe,696.619674
4,Inception,actor,Tom Hardy,399.019419
5,Inception,actor,Dileep Rao,690.564660
6,Inception,actor,Cillian Murphy,242.436096
7,Inception,actor,Tom Berenger,54.745883
8,Inception,actress,Marion Cotillard,196.976036
9,Inception,actor,Pete Postlethwaite,250.098971


In [125]:
movies_unique = movies.drop_duplicates(subset=['tconst']).copy()

movies_final = movies_unique.merge(actor_stats, on='tconst', how='left') \
    .merge(prod_stats, on='tconst', how='left')
movies_final = movies_final.drop(columns=['nconst', "tconst", "primaryName", "category"])
movies_final.head()


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,production_companies,production_countries,spoken_languages,keywords,actor_avg,actor_med,actor_dev,production_avg,production_med,production_dev
0,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",338.459844,246.267534,238.142839,1054.527486,1034.746057,39.562860
1,157336,Interstellar,8.417,32571,Released,11/5/2014,701729206,169,False,165000000,...,"Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",258.618166,154.114164,222.647814,862.125977,1034.746057,376.005072
2,155,The Dark Knight,8.512,30619,Released,7/16/2008,1004558444,152,False,185000000,...,"DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",267.919374,191.809402,210.527878,773.315771,836.477049,328.158370
3,19995,Avatar,7.573,29815,Released,12/15/2009,2923706026,162,False,237000000,...,"Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",625.067922,511.999718,322.760089,1620.716567,1241.073114,759.286907
4,24428,The Avengers,7.710,29166,Released,4/25/2012,1518815515,143,False,220000000,...,Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",883.491854,865.975453,283.026839,750.170603,566.326504,629.076898


In [126]:
movies_final.to_csv("movie-data/movies_with_scores.csv", index=False)
print("Saved movies_with_scores.csv")

Saved movies_with_scores.csv


In [127]:
print("Top 10 movies with highest actor average score:")
top_actor_avg = movies_final.sort_values('actor_avg', ascending=False).head(10)
print(top_actor_avg[['title', 'actor_avg', 'actor_med', 'actor_dev']])
print("Top 10 movies with highest production average score:")  
top_prod_avg = movies_final.sort_values('production_avg', ascending=False).head(10)
print(top_prod_avg[['title', 'production_avg', 'production_med', 'production_dev']])

Top 10 movies with highest actor average score:
                                                 title    actor_avg  \
131                                      Despicable Me  1599.311881   
27                                       Black Panther  1433.743253   
550                                    Despicable Me 3  1418.694339   
6                               Avengers: Infinity War  1390.439711   
71                   The Hobbit: An Unexpected Journey  1285.659387   
19   The Lord of the Rings: The Fellowship of the Ring  1211.470517   
15                                   Avengers: Endgame  1209.304141   
65        Harry Potter and the Deathly Hallows: Part 1  1194.981458   
129                           Star Wars: The Last Jedi  1190.508317   
179                The Hobbit: The Desolation of Smaug  1072.493614   

       actor_med    actor_dev  
131  1013.769427  1250.230198  
27   1318.442031  1142.255621  
550   800.588075  1232.272741  
6     887.291056   905.480310  
71    929.